# 17번. 기업별 충격민감도 (Rolling WLS Beta)

## 개요
기업별로 **거시경제 충격에 대한 민감도(β)** 를 추정하는 코드입니다.

$$dF_{it} = \alpha_i + \beta_i \cdot dM_t + \varepsilon_{it}$$

- $dF_{it}$: 기업 $i$의 $t$년도 매출액증가율
- $dM_t$: $t$년도 GDP 성장률 (거시 충격 대리변수)
- $\beta_i$: **충격민감도** — GDP 1%p 변화에 대한 매출증가율 반응

## 방법론 요약
| 항목 | 내용 |
|------|------|
| 추정 방법 | WLS (지수 감쇠 가중치, 최근 연도 → 높은 가중치) |
| Walk-Forward | 훈련 [2012~T], 테스트 [T+1], 최근 3개 폴드 실행 |
| Shrinkage | 관측수 기반 β 축소 (n≥5: λ=0.1, n=3~4: λ=0.4, fallback: λ=1.0) |
| S2 스코어 | 테스트연도 내 \|β\| 백분위 순위 (0~1) |


In [ ]:
import sys, io
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import warnings
warnings.filterwarnings('ignore')


## 1. 경로 및 파라미터 설정

| 파라미터 | 값 | 설명 |
|----------|----|------|
| `TRAIN_START` | 2012 | 훈련 데이터 시작 연도 |
| `MIN_TRAIN_YR` | 5 | 최소 훈련 연도 수 (5년치 확보 후 첫 테스트) |
| `MIN_FIRM_OBS` | 3 | 기업별 OLS 추정 최소 관측수 |
| `OUTLIER_CAP` | 500 | 매출증가율 이상치 상하한 (±500%) |
| `DECAY_LAMBDA` | 0.8 | 지수 감쇠율 (1년 전 → 0.8배, 2년 전 → 0.64배) |


In [ ]:
BASE      = Path().resolve().parent / '거시경제'
MACRO_CSV = BASE / '거시지표_통합.csv'
FIRM_PAR  = BASE / 'M19_도매_소매업(최종)(진).parquet'

REV_COL  = '매출액증가율'
FIRM_COL = '사업자등록번호'
YEAR_COL = '회계년도'

TRAIN_START  = 2012
MIN_TRAIN_YR = 5
MIN_FIRM_OBS = 3
OUTLIER_CAP  = 500.0
DECAY_LAMBDA = 0.8


## 2. 헬퍼 함수 정의

### 2-1. IPS 단위근 검정
Im-Pesaran-Shin 패널 단위근 검정 — 매출증가율(dF)이 단위근을 가지는지 진단합니다.
W 통계량이 음수이고 p < 0.05이면 단위근 없음(정상성) → 회귀 분석 적합.

### 2-2. Granger 인과검정
GDP 성장률(dM)이 매출증가율(dF)을 Granger 인과하는지 검정합니다.
p < 0.05이면 dM → dF 인과관계 유효 → β 추정 의미 있음.

### 2-3. WLS β 추정
`np.polyfit`에 `w=sqrt(weights)` 를 전달하여 가중 최소제곱을 수행합니다.
polyfit은 $\sum w_i^2 \cdot \varepsilon_i^2$ 를 최소화하므로 반드시 sqrt를 취해야 합니다.


In [ ]:
def ips_unit_root(panel_sub, var_col, min_obs=5):
    """Im-Pesaran-Shin 패널 단위근 검정 (진단용)"""
    t_stats = []
    for _, grp in panel_sub.groupby(FIRM_COL):
        y = grp[var_col].dropna().values
        if len(y) < min_obs:
            continue
        try:
            t, *_ = adfuller(y, maxlag=1, autolag=None)
            t_stats.append(float(t))
        except Exception:
            pass
    if len(t_stats) < 10:
        return np.nan, np.nan, len(t_stats)
    n    = len(t_stats)
    tbar = np.nanmean(t_stats)
    W    = np.sqrt(n) * (tbar - (-1.56)) / np.sqrt(1.40)
    p    = float(scipy_stats.norm.cdf(W))
    return W, p, n


def granger_bidir(ts_dm, ts_df, maxlag=1):
    """양방향 Granger 인과검정 p값 반환"""
    def _pval(y, x):
        try:
            res = grangercausalitytests(
                np.column_stack([y, x]), maxlag=maxlag, verbose=False
            )
            return float(res[maxlag][0]['ssr_ftest'][1])
        except Exception:
            return np.nan
    return _pval(ts_df, ts_dm), _pval(ts_dm, ts_df)


def estimate_wls_beta(dM_arr, dF_arr, weights):
    """WLS β 추정. polyfit에 sqrt(weights) 전달 필수."""
    try:
        coeffs = np.polyfit(dM_arr, dF_arr, 1, w=np.sqrt(weights))
        beta, alpha = float(coeffs[0]), float(coeffs[1])
        if not np.isfinite(beta):
            return None, None
        y_pred  = alpha + beta * dM_arr
        w_sum   = weights.sum()
        y_wmean = (weights * dF_arr).sum() / w_sum
        ss_res  = (weights * (dF_arr - y_pred) ** 2).sum()
        ss_tot  = (weights * (dF_arr - y_wmean) ** 2).sum()
        r2      = float(max(0.0, 1 - ss_res / ss_tot)) if ss_tot > 0 else 0.0
        return beta, r2
    except Exception:
        return None, None


## 3. 데이터 로드

- **거시지표**: GDP 명목 성장률 (`dM`) 계산
- **기업데이터**: 도소매업 기업의 매출액증가율 (`dF`)
- **패널 구성**: 두 데이터를 연도 기준으로 병합, 이상치 제거 후 분석용 패널 생성


In [ ]:
macro = pd.read_csv(MACRO_CSV, encoding='utf-8-sig')
macro['dM'] = macro['국내총생산(명목, 원화표시)'].pct_change() * 100
macro = macro[['연도', 'dM']].dropna().rename(columns={'연도': YEAR_COL})
print(f'거시지표: {len(macro)}개 연도  ({macro[YEAR_COL].min():.0f}~{macro[YEAR_COL].max():.0f})')

firm = pd.read_parquet(FIRM_PAR)
print(f'기업데이터: {len(firm):,}행  |  기업: {firm[FIRM_COL].nunique():,}개')

firm_name_map = (
    firm[[FIRM_COL, YEAR_COL, '회사명']]
    .sort_values(YEAR_COL)
    .drop_duplicates(FIRM_COL, keep='last')[[FIRM_COL, '회사명']]
)

panel = (
    firm[[FIRM_COL, YEAR_COL, REV_COL]]
    .rename(columns={REV_COL: 'dF'})
    .dropna(subset=['dF'])
    .query(f'dF >= -{OUTLIER_CAP} and dF <= {OUTLIER_CAP}')
    .merge(macro, on=YEAR_COL)
    .query(f'{YEAR_COL} >= {TRAIN_START}')
    .copy()
)
print(f'분석 패널: {len(panel):,}행  |  기업: {panel[FIRM_COL].nunique():,}개')
print(f'연도범위: {panel[YEAR_COL].min()} ~ {panel[YEAR_COL].max()}')


## 4. Walk-Forward WLS 추정

### Walk-Forward 방식이란?
- 매 폴드마다 **과거 데이터로만 훈련**, 미래(+1년)로 테스트
- 미래 데이터 누출(data leakage) 방지
- `folds[-3:]` → 2022, 2023, 2024년 테스트 폴드만 실행

### Shrinkage (수축 추정)
OLS 추정값을 전체 중앙값 β 방향으로 수축시켜 과적합 방지:

$$\hat{\beta}_i = (1-\lambda)\hat{\beta}_{OLS} + \lambda \cdot \text{median}(\beta)$$

| 관측수 | λ | 의미 |
|--------|---|------|
| n ≥ 5 | 0.1 | 데이터 충분 → OLS 신뢰 |
| n = 3~4 | 0.4 | 데이터 부족 → 중앙값 쪽으로 수축 |
| fallback | 1.0 | OLS 불가 → 전체 중앙값 사용 |


In [ ]:
all_years = sorted(panel[YEAR_COL].unique())
folds = [
    y for y in all_years
    if y >= TRAIN_START + MIN_TRAIN_YR - 1 and y + 1 in all_years
]
folds = folds[-3:]
print(f'Walk-Forward 훈련 끝 연도: {folds}')

all_scores = []

for T_k in folds:
    train_yrs = [y for y in all_years if TRAIN_START <= y <= T_k]
    test_yr   = T_k + 1
    tr = panel[panel[YEAR_COL].isin(train_yrs)].copy()
    te = panel[panel[YEAR_COL] == test_yr].copy()
    if te.empty:
        continue

    print(f'\n{"=" * 64}')
    print(f'[훈련: {TRAIN_START}~{T_k}  |  테스트: {test_yr}]')
    print(f'훈련 기업: {tr[FIRM_COL].nunique():,}개  |  테스트 기업: {te[FIRM_COL].nunique():,}개')

    W, p_ips, n_ips = ips_unit_root(tr, 'dF')
    ts_mean = (
        tr.groupby(YEAR_COL)
        .agg(dF=('dF', 'mean'), dM=('dM', 'first'))
        .sort_index().dropna()
    )
    p_gr_fwd, p_gr_bwd = granger_bidir(ts_mean['dM'].values, ts_mean['dF'].values)
    print(f'  [진단]  IPS_W={W:.3f}  IPS_p={p_ips:.3f}  n={n_ips}')
    print(f'          Granger dM->dF p={p_gr_fwd:.3f}  |  dF->dM p={p_gr_bwd:.3f}')

    firm_records = []
    for firm_id in tr[FIRM_COL].unique():
        fdata = (
            tr[tr[FIRM_COL] == firm_id]
            .sort_values(YEAR_COL)[[YEAR_COL, 'dM', 'dF']]
            .dropna()
        )
        if len(fdata) < MIN_FIRM_OBS:
            continue
        distances = test_yr - fdata[YEAR_COL].values
        weights   = DECAY_LAMBDA ** distances
        beta, r2  = estimate_wls_beta(fdata['dM'].values, fdata['dF'].values, weights)
        if beta is None:
            continue
        firm_records.append({FIRM_COL: firm_id, 'beta_i': beta, 'r2': r2, 'n_obs': len(fdata)})

    firm_df = pd.DataFrame(firm_records) if firm_records else pd.DataFrame(
        columns=[FIRM_COL, 'beta_i', 'r2', 'n_obs'])

    if not firm_df.empty:
        beta_cap  = float(firm_df['beta_i'].abs().quantile(0.99))
        n_clipped = int((firm_df['beta_i'].abs() > beta_cap).sum())
        firm_df['beta_i'] = firm_df['beta_i'].clip(-beta_cap, beta_cap)
    else:
        beta_cap, n_clipped = 0.0, 0

    n_success = len(firm_df)
    n_total   = len(tr[FIRM_COL].unique())
    print(f'  OLS 성공: {n_success:,} / {n_total:,}개 ({n_success/n_total*100:.1f}%)')
    print(f'  BETA_CAP(99분위): {beta_cap:.2f}  winsorize: {n_clipped}개')

    global_median_beta = firm_df['beta_i'].median() if not firm_df.empty else 0.0
    te_scored = te[[FIRM_COL]].drop_duplicates().merge(
        firm_df[[FIRM_COL, 'beta_i', 'r2', 'n_obs']], on=FIRM_COL, how='left')

    mask_missing            = te_scored['beta_i'].isna()
    te_scored['beta_i_raw'] = te_scored['beta_i']

    shrink_lam = pd.Series(1.0, index=te_scored.index)
    shrink_lam[~mask_missing & (te_scored['n_obs'] >= 5)] = 0.1
    shrink_lam[~mask_missing & (te_scored['n_obs'] <  5)] = 0.4

    beta_fill             = te_scored['beta_i_raw'].fillna(global_median_beta)
    te_scored['beta_i']   = (1 - shrink_lam) * beta_fill + shrink_lam * global_median_beta
    te_scored['beta_src'] = np.where(mask_missing, 'global_median', 'ols_shrunk')
    te_scored['훈련_끝']    = T_k
    te_scored['테스트_연도'] = test_yr
    te_scored['granger_p']  = round(p_gr_fwd, 4)

    n_high = int((~mask_missing & (te_scored['n_obs'] >= 5)).sum())
    n_low  = int((~mask_missing & (te_scored['n_obs'] <  5)).sum())
    print(f'  Shrinkage — high(λ=0.1): {n_high:,}  low(λ=0.4): {n_low:,}  fallback(λ=1.0): {mask_missing.sum():,}')

    all_scores.append(te_scored)


## 5. S2 스코어 산출 및 저장

### S2 스코어란?
- 테스트 연도 내 모든 기업의 |β| 를 백분위 순위로 변환 (0~1)
- 값이 높을수록 거시 충격에 민감한 기업 → 리스크 ↑
- 부호(방향)는 `beta_sign` 컬럼으로 별도 제공

### beta_quality 분류
| 값 | 조건 | 신뢰도 |
|----|------|--------|
| `high` | n_obs ≥ 5 | 높음 |
| `low` | n_obs = 3~4 | 낮음 |
| `fallback` | OLS 불가 | 전체 중앙값 대체 |


In [ ]:
if all_scores:
    scores = pd.concat(all_scores, ignore_index=True)
    scores['beta_i_abs'] = scores['beta_i'].abs()
    scores['beta_sign']  = np.sign(scores['beta_i'])
    scores['S2'] = scores.groupby('테스트_연도')['beta_i_abs'].rank(pct=True)

    scores['beta_quality'] = pd.cut(
        scores['n_obs'].fillna(0),
        bins=[0, 2, 4, 999],
        labels=['fallback', 'low', 'high']
    )

    print('S2 분포 (테스트연도별):')
    print(scores.groupby('테스트_연도')['S2'].describe().round(3))

    print('\nbeta_quality 분포:')
    print(scores.groupby(['테스트_연도','beta_quality']).size().unstack(fill_value=0))

    ols_cov = (scores['beta_src'] == 'ols_shrunk').mean()
    print(f'\nOLS 커버리지: {ols_cov:.1%}  |  fallback: {1-ols_cov:.1%}')

    scores = scores.merge(firm_name_map, on=FIRM_COL, how='left')
    col_order = [FIRM_COL, '회사명', '테스트_연도', '훈련_끝',
                 'beta_i', 'beta_i_raw', 'beta_sign', 'beta_quality', 'beta_src',
                 'S2', 'r2', 'n_obs', 'granger_p']
    col_order = [c for c in col_order if c in scores.columns]
    scores = scores[col_order]

    out_par = BASE / '충격민감도_OLS.parquet'
    out_csv = BASE / '충격민감도_OLS.csv'
    scores.to_parquet(out_par, index=False)
    scores.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f'\n저장 완료: {out_par}')
    print(f'저장 완료: {out_csv}')
    print(f'\n총 {len(scores):,}건  |  기업: {scores[FIRM_COL].nunique():,}개')
